<a href="https://colab.research.google.com/github/SorenGrubb/microns-neuroglancer-tool/blob/main/MICrONS_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests

def download_fragments(ranges, shard_url,
                       outdir="/content/neuron_drc"):

    os.makedirs(outdir, exist_ok=True)

    total = len(ranges)

    for i, (start, end) in enumerate(ranges):

        if i % 25 == 0:
            print(f"{i}/{total}")

        r = requests.get(
            shard_url,
            headers={
                "Range":
                f"bytes={start}-{end}"
            }
        )

        with open(
            f"{outdir}/{i:04d}.drc",
            "wb"
        ) as f:
            f.write(r.content)

    print("Downloaded", total, "fragments")

    return outdir

In [ ]:
!apt-get -qq install cmake > /dev/null

!git clone https://github.com/google/draco.git

!cmake -S draco -B draco/build

!cmake --build draco/build -j4

fatal: destination path 'draco' already exists and is not an empty directory.
CMake Warning (dev) at CMakeLists.txt:40 (include):
  Policy CMP0148 is not set: The FindPythonInterp and FindPythonLibs modules
  are removed.  Run "cmake --help-policy CMP0148" for policy details.  Use
  the cmake_policy command to set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

--- Running combined CXX flags test, flags: -Wno-deprecated-declarations
--- Passed combined CXX flags test
-- Configuring done (0.1s)
-- Generating done (0.1s)
-- Build files have been written to: /content/draco/build
[  0%] Built target draco_compression_attributes_pred_schemes_dec
[  3%] Built target draco_attributes
[  8%] Built target draco_compression_attributes_dec
[ 13%] Built target draco_compression_attributes_enc
[ 14%] Built target draco_compression_attributes_pred_schemes_enc
[ 14%] Built target draco_enc_config
[ 14%] Built target draco_dec_config
[ 20%]

In [ ]:
import subprocess
import os

def decode_fragments(
    drc_dir="/content/neuron_drc",
    obj_dir="/content/neuron_obj"
):

    os.makedirs(obj_dir, exist_ok=True)

    decoder = (
        "/content/draco/build/draco_decoder"
    )

    success = 0

    for file in sorted(os.listdir(drc_dir)):

        if not file.endswith(".drc"):
            continue

        inp = f"{drc_dir}/{file}"

        out = (
            f"{obj_dir}/"
            + file.replace(".drc", ".obj")
        )

        result = subprocess.run(
            [
                decoder,
                "-i", inp,
                "-o", out
            ],
            capture_output=True
        )

        if result.returncode == 0:
            success += 1

    print("Decoded:", success)

    return obj_dir

In [ ]:
!pip -q install trimesh

import trimesh
import numpy as np
import os

def reconstruct_lod0(
    parsed,
    obj_dir="/content/neuron_obj",
    output_glb="/content/neuron.glb"
):

    chunk_shape = np.array(
        parsed["chunk_shape"],
        dtype=np.float64
    )

    grid_origin = np.array(
        parsed["grid_origin"],
        dtype=np.float64
    )

    vertex_offsets = parsed[
        "vertex_offsets"
    ]

    positions = (
        parsed["lods"][0]["positions"]
    )

    n = int(
        parsed["num_fragments_per_lod"][0]
    )

    # MICrONS voxel size (nm)
    VOXEL_SIZE = np.array(
        [8.0, 8.0, 40.0],
        dtype=np.float64
    )

    meshes = []

    for i in range(n):

        path = f"{obj_dir}/{i:04d}.obj"

        if not os.path.exists(path):
            continue

        mesh = trimesh.load(
            path,
            force="mesh"
        )

        pos = np.array([
            positions[0, i],
            positions[1, i],
            positions[2, i]
        ], dtype=np.float64)

        verts = mesh.vertices.astype(
            np.float64
        )

        # Draco stores vertices as uint16
        verts /= 65535.0

        verts = (
            grid_origin
            + vertex_offsets[0]
            + chunk_shape * (pos + verts)
        )

        # Convert voxel coordinates to physical units
        verts *= VOXEL_SIZE

        mesh.vertices = verts

        meshes.append(mesh)

    combined = trimesh.util.concatenate(
        meshes
    )

    bbox = combined.bounds
    size = bbox[1] - bbox[0]

    print("\nBounding box size:")
    print("X =", size[0])
    print("Y =", size[1])
    print("Z =", size[2])

    print("\nMesh:")
    print(combined)

    combined.export(output_glb)

    print(
        "\nSaved:",
        output_glb
    )

    return output_glb

In [ ]:
def build_fragment_ranges(
    parsed,
    manifest_start
):

    fragment_sizes = []

    for lod in parsed["lods"]:

        fragment_sizes.extend(
            lod["sizes"].tolist()
        )

    fragment_region_start = (
        manifest_start
        - sum(fragment_sizes)
    )

    ranges = []

    offset = fragment_region_start

    for size in fragment_sizes:

        start = offset
        end = offset + size - 1

        ranges.append(
            (start, end)
        )

        offset += size

    return ranges

In [ ]:
def extract_neuron(root_id):

    print(
        "Finding manifest..."
    )

    info = find_manifest(root_id)

    manifest = download_manifest(info)

    parsed = parse_manifest(
        manifest
    )

    ranges = build_fragment_ranges(
        parsed,
        info["manifest_start"]
    )

    print(
        "Fragments:",
        len(ranges)
    )

    download_fragments(
        ranges,
        info["shard_url"]
    )

    decode_fragments()

    glb = reconstruct_lod0(
        parsed,
        output_glb=f"/content/{root_id}.glb"
    )

    return glb

In [ ]:
import requests
import gzip
import struct
import numpy as np

def find_manifest(root_id):

    shard, minishard = get_shard_and_minishard(root_id)

    shard_url = (
        "https://storage.googleapis.com/storage/v1/b/"
        "iarpa_microns/o/"
        f"minnie%2Fminnie65%2Fseg_m1300%2Fmesh%2F{shard}.shard"
        "?alt=media"
    )

    minishard_bits = 8

    shard_index_offset = minishard * 16

    r = requests.get(
        shard_url,
        headers={
            "Range":
            f"bytes={shard_index_offset}-{shard_index_offset+15}"
        }
    )

    entry = r.content

    start_offset = struct.unpack("<Q", entry[:8])[0]
    end_offset   = struct.unpack("<Q", entry[8:])[0]

    shard_index_size = (2 ** minishard_bits) * 16

    minishard_start = shard_index_size + start_offset
    minishard_end   = shard_index_size + end_offset

    r = requests.get(
        shard_url,
        headers={
            "Range":
            f"bytes={minishard_start}-{minishard_end-1}"
        }
    )

    decoded = gzip.decompress(r.content)

    arr = np.frombuffer(
        decoded,
        dtype="<u8"
    )

    n = len(arr) // 3

    ids = arr[:n].copy()
    starts = arr[n:2*n].copy()
    sizes = arr[2*n:].copy()

    for i in range(1, n):
        ids[i] += ids[i-1]

    prev_start = shard_index_size

    for i in range(n):
        starts[i] += prev_start
        prev_start = starts[i] + sizes[i]

    matches = np.where(ids == root_id)[0]

    if len(matches) == 0:
        raise ValueError(
            f"Root ID {root_id} not found"
        )

    idx = int(matches[0])

    manifest_start = int(starts[idx])
    manifest_size  = int(sizes[idx])
    manifest_end   = manifest_start + manifest_size

    return {
        "shard": shard,
        "minishard": minishard,
        "manifest_start": manifest_start,
        "manifest_end": manifest_end,
        "manifest_size": manifest_size,
        "shard_url": shard_url
    }

In [ ]:
for name in [
    "get_shard_and_minishard",
    "find_manifest",
    "download_manifest",
    "parse_manifest",
    "extract_neuron"
]:
    print(name, "=>", name in globals())

get_shard_and_minishard => True
find_manifest => True
download_manifest => True
parse_manifest => True
extract_neuron => True


In [ ]:
def rotl32(x, r):
    x &= 0xffffffff
    return ((x << r) | (x >> (32 - r))) & 0xffffffff

def murmur_mix(h):
    h ^= (h >> 16)
    h = (h * 0x85ebca6b) & 0xffffffff
    h ^= (h >> 13)
    h = (h * 0xc2b2ae35) & 0xffffffff
    h ^= (h >> 16)
    return h & 0xffffffff

def murmurHash3_x86_128Hash64Bits_Bigint(seed, input_val):

    h1 = seed
    h2 = seed
    h3 = seed
    h4 = seed

    c1 = 0x239b961b
    c2 = 0xab0e9789
    c3 = 0x38b34ae5

    high32 = (input_val >> 32) & 0xffffffff
    low32 = input_val & 0xffffffff

    k2 = (high32 * c2) & 0xffffffff
    k2 = rotl32(k2, 16)
    k2 = (k2 * c3) & 0xffffffff
    h2 ^= k2

    k1 = (low32 * c1) & 0xffffffff
    k1 = rotl32(k1, 15)
    k1 = (k1 * c2) & 0xffffffff
    h1 ^= k1

    length = 8

    h1 ^= length
    h2 ^= length
    h3 ^= length
    h4 ^= length

    h1 = (h1 + h2 + h3 + h4) & 0xffffffff
    h2 = (h2 + h1) & 0xffffffff
    h3 = (h3 + h1) & 0xffffffff
    h4 = (h4 + h1) & 0xffffffff

    h1 = murmur_mix(h1)
    h2 = murmur_mix(h2)
    h3 = murmur_mix(h3)
    h4 = murmur_mix(h4)

    h1 = (h1 + h2 + h3 + h4) & 0xffffffff
    h2 = (h2 + h1) & 0xffffffff

    return h1 | (h2 << 32)

def get_shard_and_minishard(root_id):

    hashed = murmurHash3_x86_128Hash64Bits_Bigint(
        0,
        root_id >> 6
    )

    minishard_bits = 8
    shard_bits = 13

    mask = (1 << (minishard_bits + shard_bits)) - 1

    shard_and_minishard = hashed & mask

    minishard = (
        shard_and_minishard &
        ((1 << minishard_bits) - 1)
    )

    shard = (
        (shard_and_minishard >> minishard_bits)
        &
        ((1 << shard_bits) - 1)
    )

    return format(shard, "04x"), minishard

In [ ]:
import requests
import gzip

def download_manifest(info):

    r = requests.get(
        info["shard_url"],
        headers={
            "Range":
            f"bytes={info['manifest_start']}-{info['manifest_end']-1}"
        }
    )

    return gzip.decompress(r.content)

In [ ]:
for name in [
    "get_shard_and_minishard",
    "find_manifest",
    "download_manifest",
    "parse_manifest",
    "extract_neuron"
]:
    print(name, "=>", name in globals())

get_shard_and_minishard => True
find_manifest => True
download_manifest => True
parse_manifest => True
extract_neuron => True


In [ ]:
import struct
import numpy as np

def parse_manifest(manifest):

    ptr = 0

    chunk_shape = struct.unpack_from(
        "<3f",
        manifest,
        ptr
    )
    ptr += 12

    grid_origin = struct.unpack_from(
        "<3f",
        manifest,
        ptr
    )
    ptr += 12

    num_lods = struct.unpack_from(
        "<I",
        manifest,
        ptr
    )[0]

    ptr += 4

    lod_scales = struct.unpack_from(
        f"<{num_lods}f",
        manifest,
        ptr
    )

    ptr += 4 * num_lods

    vertex_offsets = np.frombuffer(
        manifest,
        dtype="<f4",
        count=num_lods * 3,
        offset=ptr
    ).reshape(num_lods, 3)

    ptr += num_lods * 3 * 4

    num_fragments_per_lod = struct.unpack_from(
        f"<{num_lods}I",
        manifest,
        ptr
    )

    ptr += num_lods * 4

    lods = []

    for lod in range(num_lods):

        n = num_fragments_per_lod[lod]

        positions = np.frombuffer(
            manifest,
            dtype="<u4",
            count=n * 3,
            offset=ptr
        ).reshape(3, n)

        ptr += n * 3 * 4

        sizes = np.frombuffer(
            manifest,
            dtype="<u4",
            count=n,
            offset=ptr
        )

        ptr += n * 4

        lods.append({
            "positions": positions,
            "sizes": sizes
        })

    return {
        "chunk_shape": chunk_shape,
        "grid_origin": grid_origin,
        "lod_scales": lod_scales,
        "vertex_offsets": vertex_offsets,
        "num_fragments_per_lod": num_fragments_per_lod,
        "lods": lods
    }

In [ ]:
print("parse_manifest" in globals())

True


In [ ]:
for name in [
    "get_shard_and_minishard",
    "find_manifest",
    "download_manifest",
    "parse_manifest",
    "build_fragment_ranges",
    "download_fragments",
    "decode_fragments",
    "reconstruct_lod0",
    "extract_neuron"
]:
    print(name, "=>", name in globals())

get_shard_and_minishard => True
find_manifest => True
download_manifest => True
parse_manifest => True
build_fragment_ranges => True
download_fragments => True
decode_fragments => True
reconstruct_lod0 => True
extract_neuron => True


In [57]:
ROOT_ID = 864691135278214049

glb_file = extract_neuron(ROOT_ID)

print(glb_file)

Finding manifest...
Fragments: 18
0/18
Downloaded 18 fragments
Decoded: 533

Bounding box size:
X = 9296.01684596017
Y = 13979.963317311369
Z = 18239.96581979096

Mesh:
<trimesh.Trimesh(vertices.shape=(213671, 3), faces.shape=(426180, 3))>

Saved: /content/864691135278214049.glb
/content/864691135278214049.glb
